# wav2vec2 Fine-Tuning

This notebook is set up to run end-to-end on a Google Colab runtime.

When it detects Colab, it will:
- clone the GitHub repo into `/content/speech-emotion-directions`
- install any missing training dependencies
- download the official RAVDESS speech archive
- rebuild the metadata CSV with Colab file paths
- launch the `facebook/wav2vec2-base` fine-tuning run

When it is run locally, it uses the current project folder instead.

In [1]:
from __future__ import annotations

import importlib.util
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"
RAVDESS_SOURCES = [
    {
        "name": "RAVDESS Speech 16K",
        "url": "https://zenodo.org/api/records/11063852/files/Audio_Speech_Actors_01-24_16k.zip/content",
        "archive_name": "Audio_Speech_Actors_01-24_16k.zip",
    },
    {
        "name": "RAVDESS Speech Original",
        "url": "https://zenodo.org/api/records/1188976/files/Audio_Speech_Actors_01-24.zip/content",
        "archive_name": "Audio_Speech_Actors_01-24.zip",
    },
]

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)


def run_command(cmd: list[str], cwd: Path | None = None) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)



def ensure_colab_packages() -> None:
    if not IS_COLAB:
        return

    package_map = {
        "yaml": "pyyaml",
        "pandas": "pandas",
        "numpy": "numpy",
        "soundfile": "soundfile",
        "librosa": "librosa",
        "torch": "torch",
        "torchaudio": "torchaudio",
        "transformers": "transformers",
        "datasets": "datasets",
        "accelerate": "accelerate",
        "sklearn": "scikit-learn",
        "tqdm": "tqdm",
    }
    missing_packages = sorted({pkg for module, pkg in package_map.items() if importlib.util.find_spec(module) is None})
    if missing_packages:
        run_command([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
    else:
        print("Required training packages are already available.")



def find_local_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / "FinalProject"]

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "configs" / "wav2vec.yaml").exists() and (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root locally. Expected a folder containing configs/wav2vec.yaml and src/."
    )



def repo_status_lines(repo_root: Path) -> list[str]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "status", "--short"],
        text=True,
        capture_output=True,
        check=True,
    )
    return [line for line in result.stdout.splitlines() if line.strip()]



def repo_ahead_behind(repo_root: Path) -> tuple[int, int]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "rev-list", "--left-right", "--count", "HEAD...origin/main"],
        text=True,
        capture_output=True,
        check=False,
    )
    if result.returncode != 0 or not result.stdout.strip():
        return 0, 0
    ahead_str, behind_str = result.stdout.strip().split()
    return int(ahead_str), int(behind_str)



def clone_clean_code_checkout(clean_root: Path) -> Path:
    if clean_root.exists():
        shutil.rmtree(clean_root)
    run_command(["git", "clone", "--depth", "1", REPO_URL, str(clean_root)])
    return clean_root



def prepare_project_and_code_roots() -> tuple[Path, Path]:
    runtime_root = Path("/content") / REPO_NAME
    clean_root = Path("/content") / f"{REPO_NAME}-clean"

    if runtime_root.exists() and (runtime_root / ".git").exists():
        try:
            run_command(["git", "-C", str(runtime_root), "fetch", "origin"])
        except subprocess.CalledProcessError as exc:
            print(f"Fetch failed for existing Colab repo; continuing with local state. Details: {exc}")

        status_lines = repo_status_lines(runtime_root)
        ahead, behind = repo_ahead_behind(runtime_root)

        if not status_lines and ahead == 0:
            if behind > 0:
                try:
                    run_command(["git", "-C", str(runtime_root), "pull", "--ff-only", "origin", "main"])
                    code_root = runtime_root
                except subprocess.CalledProcessError as exc:
                    print(f"Fast-forward pull failed; using a clean code checkout instead. Details: {exc}")
                    code_root = clone_clean_code_checkout(clean_root)
            else:
                code_root = runtime_root
        else:
            print(f"Using a clean code checkout because {runtime_root} has local changes or local commits.")
            for line in status_lines[:10]:
                print(" -", line)
            if ahead or behind:
                print(f"Repo divergence relative to origin/main: ahead={ahead}, behind={behind}")
            code_root = clone_clean_code_checkout(clean_root)

        project_root = runtime_root
    elif runtime_root.exists():
        print(f"Using existing non-git project directory for data/artifacts: {runtime_root}")
        project_root = runtime_root
        code_root = clone_clean_code_checkout(clean_root)
    else:
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(runtime_root)])
        project_root = runtime_root
        code_root = runtime_root

    return project_root, code_root


ensure_colab_packages()

if IS_COLAB:
    PROJECT_ROOT, CODE_ROOT = prepare_project_and_code_roots()
else:
    PROJECT_ROOT = find_local_project_root()
    CODE_ROOT = PROJECT_ROOT

os.chdir(PROJECT_ROOT)
for root in [CODE_ROOT, PROJECT_ROOT]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

print("Project root:", PROJECT_ROOT)
print("Code root:", CODE_ROOT)
print("Working directory:", Path.cwd().resolve())

Running in Colab: True
Required training packages are already available.
+ git -C /content/speech-emotion-directions pull --ff-only origin main
Project root: /content/speech-emotion-directions
Working directory: /content/speech-emotion-directions


In [2]:
import pandas as pd
from IPython.display import display

from huggingface_hub import snapshot_download

from src.data.ravdess_metadata import build_ravdess_metadata, save_metadata

raw_root = PROJECT_ROOT / "data" / "raw"
extract_dir = raw_root / "ravdess_audio_speech"
metadata_path = PROJECT_ROOT / "data" / "metadata" / "ravdess_metadata.csv"

raw_root.mkdir(parents=True, exist_ok=True)
metadata_path.parent.mkdir(parents=True, exist_ok=True)



def download_file(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        destination.unlink()

    commands = [
        [
            "curl",
            "-L",
            "--fail",
            "--retry",
            "3",
            "--retry-delay",
            "5",
            "-A",
            "Mozilla/5.0",
            "-o",
            str(destination),
            url,
        ],
        [
            "wget",
            "--tries=3",
            "--waitretry=5",
            "--user-agent=Mozilla/5.0",
            "-O",
            str(destination),
            url,
        ],
    ]

    last_error = None
    for command in commands:
        if shutil.which(command[0]) is None:
            continue
        try:
            print("+", " ".join(command))
            subprocess.check_call(command)
            if destination.exists() and destination.stat().st_size > 0:
                return
        except subprocess.CalledProcessError as exc:
            last_error = exc
            if destination.exists():
                destination.unlink()

    raise RuntimeError(f"Failed to download dataset from {url}") from last_error


def ensure_ravdess_archive(raw_root: Path) -> Path:
    last_error = None
    for source in RAVDESS_SOURCES:
        archive_path = raw_root / source["archive_name"]
        if archive_path.exists() and archive_path.stat().st_size > 0:
            print(f"Using existing archive: {archive_path}")
            return archive_path

        try:
            print(f"Downloading {source['name']}...")
            download_file(source["url"], archive_path)
            print(f"Downloaded archive to: {archive_path}")
            return archive_path
        except RuntimeError as exc:
            print(f"Download failed for {source['name']}: {exc}")
            last_error = exc

    raise RuntimeError("Unable to download any RAVDESS speech archive in this runtime.") from last_error


def download_from_hf_mirror(destination_root: Path) -> None:
    print("Falling back to the Hugging Face RAVDESS mirror...")
    snapshot_download(
        repo_id="birgermoell/ravdess",
        repo_type="dataset",
        local_dir=destination_root,
        allow_patterns=["Actor_*/*.wav"],
    )



def count_wavs(root: Path) -> int:
    if not root.exists():
        return 0
    return sum(1 for _ in root.rglob("*.wav"))


wav_count = count_wavs(extract_dir)
print("Existing extracted wav files:", wav_count)

if wav_count < 1440:
    archive_downloaded = False
    try:
        archive_path = ensure_ravdess_archive(raw_root)
        archive_downloaded = True
    except RuntimeError as exc:
        print(exc)

    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    if archive_downloaded:
        print("Extracting archive...")
        with zipfile.ZipFile(archive_path, "r") as archive:
            archive.extractall(extract_dir)
    else:
        download_from_hf_mirror(extract_dir)

wav_count = count_wavs(extract_dir)
print("Extracted wav files:", wav_count)
assert wav_count == 1440, f"Expected 1440 speech wav files, found {wav_count}."

df = build_ravdess_metadata(extract_dir)
save_metadata(df, metadata_path)

project_df = df[df["keep_for_project"]].copy()
print("Metadata saved to:", metadata_path)
print("Total speech clips:", len(df))
print("Project clips:", len(project_df))
display(project_df["final_label"].value_counts().sort_index().rename("count").to_frame())
display(project_df.groupby(["split", "final_label"]).size().unstack(fill_value=0))

Existing extracted wav files: 0
+ curl -L --fail --retry 3 --retry-delay 5 -A Mozilla/5.0 -o /content/speech-emotion-directions/data/raw/Audio_Speech_Actors_01-24_16k.zip https://zenodo.org/api/records/11063852/files/Audio_Speech_Actors_01-24_16k.zip/content
+ wget --tries=3 --waitretry=5 --user-agent=Mozilla/5.0 -O /content/speech-emotion-directions/data/raw/Audio_Speech_Actors_01-24_16k.zip https://zenodo.org/api/records/11063852/files/Audio_Speech_Actors_01-24_16k.zip/content
Download failed for RAVDESS Speech 16K: Failed to download dataset from https://zenodo.org/api/records/11063852/files/Audio_Speech_Actors_01-24_16k.zip/content
+ curl -L --fail --retry 3 --retry-delay 5 -A Mozilla/5.0 -o /content/speech-emotion-directions/data/raw/Audio_Speech_Actors_01-24.zip https://zenodo.org/api/records/1188976/files/Audio_Speech_Actors_01-24.zip/content
+ wget --tries=3 --waitretry=5 --user-agent=Mozilla/5.0 -O /content/speech-emotion-directions/data/raw/Audio_Speech_Actors_01-24.zip https

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

Extracted wav files: 1440
Metadata saved to: /content/speech-emotion-directions/data/metadata/ravdess_metadata.csv
Total speech clips: 1440
Project clips: 1248


,count
final_label,
angry,192
disgust,192
fearful,192
happy,192
neutral,288
sad,192


final_label,angry,disgust,fearful,happy,neutral,sad
split,,,,,,
test,32,32,32,32,48,32
train,128,128,128,128,192,128
val,32,32,32,32,48,32


In [3]:
import json

import torch

from src.utils.config import load_yaml_config

config_path = PROJECT_ROOT / "configs" / "wav2vec.yaml"
config = load_yaml_config(config_path)

config["metadata_path"] = str(metadata_path.resolve())
config["output_dir"] = str((PROJECT_ROOT / config["output_dir"]).resolve())

if IS_COLAB and torch.cuda.is_available():
    config["batch_size"] = 8
    config["eval_batch_size"] = 16
    config["num_workers"] = 2

output_dir = Path(config["output_dir"])
output_dir.parent.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device_summary = torch.cuda.get_device_name(0)
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_summary = "Apple Metal (MPS)"
else:
    device_summary = "CPU"

assert config_path.exists(), f"Config file not found: {config_path}"
assert metadata_path.exists(), f"Metadata file not found: {metadata_path}"

print("Config file:", config_path)
print("Metadata file:", metadata_path)
print("Output directory:", output_dir)
print("Detected device:", device_summary)
print(json.dumps(config, indent=2))

Config file: /content/speech-emotion-directions/configs/wav2vec.yaml
Metadata file: /content/speech-emotion-directions/data/metadata/ravdess_metadata.csv
Output directory: /content/speech-emotion-directions/artifacts/checkpoints/wav2vec_ravdess_speaker_independent_v1
Detected device: NVIDIA A100-SXM4-40GB
{
  "experiment_name": "wav2vec_ravdess_speaker_independent_v1",
  "metadata_path": "/content/speech-emotion-directions/data/metadata/ravdess_metadata.csv",
  "output_dir": "/content/speech-emotion-directions/artifacts/checkpoints/wav2vec_ravdess_speaker_independent_v1",
  "backbone_name": "facebook/wav2vec2-base",
  "sample_rate": 16000,
  "batch_size": 8,
  "eval_batch_size": 16,
  "num_workers": 2,
  "learning_rate": 2e-05,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "num_epochs": 12,
  "early_stopping_patience": 4,
  "gradient_accumulation_steps": 1,
  "dropout": 0.2,
  "seed": 42,
  "freeze_feature_encoder": false,
  "freeze_feature_encoder_epochs": 1,
  "use_class_weights":

In [4]:
from src.training.train_wav2vec import run_training

run_training(config, dry_run=False)

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

...

epoch=9 train_loss=0.0457 val_accuracy=0.7981 val_macro_f1=0.7897


epoch=10 train_loss=0.0370 val_accuracy=0.8029 val_macro_f1=0.7930


epoch=11 train_loss=0.0297 val_accuracy=0.8029 val_macro_f1=0.7977


epoch=12 train_loss=0.0314 val_accuracy=0.7981 val_macro_f1=0.7891


Saved model and evaluation artifacts to /content/speech-emotion-directions/artifacts/checkpoints/wav2vec_ravdess_speaker_independent_v1
Final val macro F1: 0.7977
Final test macro F1: 0.7463


In [5]:
import json

print("Output directory:", output_dir)
print("Exists:", output_dir.exists())

if output_dir.exists():
    for path in sorted(output_dir.iterdir()):
        print(path.name)

for metrics_name in ["val_metrics.json", "test_metrics.json"]:
    metrics_path = output_dir / metrics_name
    if metrics_path.exists():
        with metrics_path.open("r", encoding="utf-8") as handle:
            metrics = json.load(handle)
        summary = {
            "accuracy": metrics.get("accuracy"),
            "macro_f1": metrics.get("macro_f1"),
            "weighted_f1": metrics.get("weighted_f1"),
        }
        print(f"\n{metrics_name}")
        print(json.dumps(summary, indent=2))

Output directory: /content/speech-emotion-directions/artifacts/checkpoints/wav2vec_ravdess_speaker_independent_v1
Exists: True
config.json
feature_extractor
label_mapping.json
model_state.pt
test_classification_report.csv
test_metrics.json
test_predictions.csv
val_metrics.json

val_metrics.json
{
  "accuracy": 0.8028846153846154,
  "macro_f1": 0.7977256110324631,
  "weighted_f1": 0.8043782968234882
}

test_metrics.json
{
  "accuracy": 0.7596153846153846,
  "macro_f1": 0.7462614987684475,
  "weighted_f1": 0.7552217756354298
}


## Optional Google Drive Sync

This sync step is the recommended way to preserve the full training outputs from Colab.

It copies the complete fine-tuned checkpoint directory, the metadata CSV, and a short run summary into your Google Drive under:

`MyDrive/speech-emotion-directions/runs/<experiment_name>/`

That makes it easy to keep the large model artifacts without trying to force them into GitHub.

In [7]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

if not IS_COLAB:
    print("Google Drive sync is intended for Colab runtimes. Skipping on local execution.")
else:
    from google.colab import drive  # type: ignore

    drive.mount('/content/drive', force_remount=False)

    drive_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' / config['experiment_name']
    checkpoint_backup_dir = drive_root / 'checkpoint'
    metadata_backup_dir = drive_root / 'metadata'

    assert output_dir.exists(), f"Training output directory does not exist: {output_dir}"
    assert metadata_path.exists(), f"Metadata CSV does not exist: {metadata_path}"

    if checkpoint_backup_dir.exists():
        shutil.rmtree(checkpoint_backup_dir)
    checkpoint_backup_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(output_dir, checkpoint_backup_dir)

    metadata_backup_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(metadata_path, metadata_backup_dir / metadata_path.name)

    val_metrics = {}
    test_metrics = {}
    if (output_dir / 'val_metrics.json').exists():
        val_metrics = json.loads((output_dir / 'val_metrics.json').read_text(encoding='utf-8'))
    if (output_dir / 'test_metrics.json').exists():
        test_metrics = json.loads((output_dir / 'test_metrics.json').read_text(encoding='utf-8'))

    summary_lines = [
        f"# {config['experiment_name']}",
        "",
        f"Generated at: {datetime.now(timezone.utc).isoformat()}",
        f"Colab project root: {PROJECT_ROOT}",
        f"Checkpoint source: {output_dir}",
        f"Drive backup root: {drive_root}",
        "",
        "## Validation Metrics",
        f"- Accuracy: {val_metrics.get('accuracy')}",
        f"- Macro F1: {val_metrics.get('macro_f1')}",
        f"- Weighted F1: {val_metrics.get('weighted_f1')}",
        "",
        "## Test Metrics",
        f"- Accuracy: {test_metrics.get('accuracy')}",
        f"- Macro F1: {test_metrics.get('macro_f1')}",
        f"- Weighted F1: {test_metrics.get('weighted_f1')}",
        "",
        "This Drive backup contains the full fine-tuned checkpoint and metadata used for the run.",
    ]
    summary_path = drive_root / 'run_summary.md'
    summary_path.write_text("\n".join(summary_lines) + "\n", encoding='utf-8')

    print('Backed up training artifacts to Google Drive:')
    print(' -', checkpoint_backup_dir)
    print(' -', metadata_backup_dir / metadata_path.name)
    print(' -', summary_path)

    print('\nDrive backup contents:')
    for path in sorted(drive_root.rglob('*')):
        relative = path.relative_to(drive_root)
        if path.is_dir():
            print(f' [dir]  {relative}')
        else:
            print(f' [file] {relative}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Backed up training artifacts to Google Drive:
 - /content/drive/MyDrive/speech-emotion-directions/runs/wav2vec_ravdess_speaker_independent_v1/checkpoint
 - /content/drive/MyDrive/speech-emotion-directions/runs/wav2vec_ravdess_speaker_independent_v1/metadata/ravdess_metadata.csv
 - /content/drive/MyDrive/speech-emotion-directions/runs/wav2vec_ravdess_speaker_independent_v1/run_summary.md

Drive backup contents:
 [dir]  checkpoint
 [file] checkpoint/config.json
 [dir]  checkpoint/feature_extractor
 [file] checkpoint/feature_extractor/preprocessor_config.json
 [file] checkpoint/label_mapping.json
 [file] checkpoint/model_state.pt
 [file] checkpoint/test_classification_report.csv
 [file] checkpoint/test_metrics.json
 [file] checkpoint/test_predictions.csv
 [file] checkpoint/val_metrics.json
 [dir]  metadata
 [file] metadata/ravdess_metadata.csv
 [file] run_summar